In [0]:
%run ./00_config

In [0]:
%run ./02_core_functions

# 03_main
Run this notebook top to bottom. For every active row in the control table it will:

1. **Copy** all zip files from the source to `<zip_tgt_path>/<mid_date>/`
2. **Unzip** them into `<unzip_tgt_path>/<mid_date>/<zip_name>/`
3. **Ingest** the JSON files into the target table, tagging each row with the `zip_file` it came from

There is no "already processed" check: every run processes everything the control table points to. Re-running is safe - the same file replaces its previous row instead of adding a duplicate.

In [0]:
run_id          = str(uuid.uuid4())
control_entries = get_active_control_entries()
ctl_by_sno      = {c["s_no"]: c for c in control_entries}
log_records     = []

print(f"run_id = {run_id}")
print(f"{len(control_entries)} active control row(s)")

### Stage 1 - Copy

In [0]:
for ctl in control_entries:
    print(f"--- s_no={ctl['s_no']} | {ctl['zip_source_path']} ---")

    all_zips = list_zip_files(ctl["zip_source_path"])
    zips = filter_zip_files_by_load_type(all_zips, ctl["is_full_load"], ctl["is_incremental_load"])
    print(f"{len(all_zips)} zip file(s) at source, {len(zips)} selected to copy")

    if ctl["is_incremental_load"] == 1 and not zips:
        now = datetime.now()
        rec = {"s_no": ctl["s_no"], "stage": "COPY", "source": ctl["zip_source"], "file_name": None,
               "source_path": ctl["zip_source_path"], "target_path": None, "status": "FAILED",
               "message": f"Incremental load: no zip file found for today's date {now:%Y%m%d}",
               "start_time": now, "end_time": now}
        log_records.append(rec)
        print(f"  [COPY:FAILED] {rec['message']}")
        continue

    for f in zips:
        rec = copy_zip_file(f, ctl["zip_tgt_path"], ctl["s_no"], ctl["zip_source"])
        log_records.append(rec)
        print(f"  [COPY:{rec['status']}] {rec['file_name']}: {rec['message']}")

### Stage 2 - Unzip

In [0]:
copied = [r for r in log_records if r["stage"] == "COPY" and r["status"] == "SUCCESS"]
print(f"{len(copied)} zip file(s) to unzip")

for copy_rec in copied:
    rec = unzip_zip_file(copy_rec, ctl_by_sno[copy_rec["s_no"]]["unzip_tgt_path"])
    log_records.append(rec)
    print(f"  [UNZIP:{rec['status']}] {rec['file_name']}: {rec['message']}")

write_logs(log_records, run_id)
print(f"Logged {len(log_records)} COPY/UNZIP record(s) to {log_table}")

### Stage 3 - Ingest JSON

In [0]:
ingest_summary = {"SUCCESS": 0, "FAILED": 0}

unzipped = [r for r in log_records if r["stage"] == "UNZIP" and r["status"] == "SUCCESS"]
print(f"{len(unzipped)} unzipped zip file(s) to ingest")

for unzip_rec in unzipped:
    print(f"--- {unzip_rec['file_name']} ---")
    counts = ingest_unzipped_zip(unzip_rec, ctl_by_sno[unzip_rec["s_no"]], run_id)
    for k, v in counts.items():
        ingest_summary[k] += v

print("\nJSON ingestion summary:", ingest_summary)

### Run summary

In [0]:
%sql
SELECT * FROM iics.casemix.zip_files_and_json_control_test

In [0]:
%sql
SELECT * FROM iics.casemix.zip_files_execution_log_test

In [0]:
%sql
SELECT * FROM iics.casemix.json_large_files_load_log

In [0]:
%sql
SELECT * FROM iics.casemix.json_test_1